In [1]:
import os
import sys
import numpy as np
import pandas as pd
from pathlib import Path
import umap
import hdbscan
import random
import matplotlib.pyplot as plt
import pickle
from tqdm.notebook import tqdm

In [2]:
cur_dir = os.getcwd().split('\\')

if cur_dir[-1] == 'notebooks':
    os.chdir("..")

from utils.data_load_utilities.data_loader import load_model_results
from utils.get_global_const import get_global_const
from utils.get_metrics import get_metrics
from utils.get_ranks import get_ranks_s, get_ranks
from methods.ADoE_method import *
from methods.k_nearest_methods import *
from methods.kmeans_methods import *
from methods.opt_methods import *
from methods.sparce_methods import *
from methods.entrophy_metods import *

from testing_pipeline.testing_pipeline_stats import *

from sklearn.linear_model import LassoCV
from sklearn.feature_selection import mutual_info_regression

from datetime import datetime
import re

import warnings
warnings.simplefilter('ignore')

In [3]:
scores, datasets, models = get_global_const()
scores

Parsing Bakeoff2017 models...

Parsing Bakeoff2021 models...

Parsing Bakeoff2023 models...

Parsing HIVE-COTEV2 models...



{'1NN-DTW':                   folds:         0         1         2         3         4  \
 0                  Adiac  0.603581  0.603581  0.601023  0.606138  0.608696   
 1              ArrowHead  0.702857  0.742857  0.680000  0.708571  0.760000   
 2                   Beef  0.633333  0.533333  0.533333  0.500000  0.566667   
 3              BeetleFly  0.700000  0.800000  0.650000  0.800000  0.650000   
 4            BirdChicken  0.750000  0.850000  0.950000  0.800000  0.950000   
 ..                   ...       ...       ...       ...       ...       ...   
 107  EOGHorizontalSignal  0.441989  0.607735  0.566298  0.569061  0.624309   
 108    EOGVerticalSignal  0.433702  0.511050  0.549724  0.566298  0.563536   
 109                 Rock  0.540000  0.680000  0.660000  0.640000  0.720000   
 110                 Crop  0.710536  0.713631  0.706964  0.708929  0.712321   
 111            Chinatown  0.973761  0.909621  0.953353  0.944606  0.970845   
 
             5         6         7     

In [4]:
# random.seed(42)
# np.random.seed(42)

In [5]:
# chosen_datasets = random.sample(datasets, 30)
# chosen_models_idx = random.sample(range(len(list(scores.keys()))), 9)
# chosen_scores = {k: scores[k] for k in np.array(models)[chosen_models_idx]}
# chosen_datasets = np.array(chosen_datasets)
# ranks = get_ranks_s(chosen_datasets, scores, datasets, return_ranks=True, model_indx=chosen_models_idx)
# ranks_all = get_ranks_s(chosen_datasets, scores, datasets, return_ranks=False, model_indx=chosen_models_idx)

In [6]:
# ranks_all

In [7]:
# for i in range(100):
#     chosen_datasets = random.sample(datasets, 30)
#     chosen_models = random.sample(models, 9)
#     chosen_scores = {k: scores[k] for k in chosen_models}
#     chosen_datasets = np.array(chosen_datasets)
#     chosen_ranks = get_ranks_s(chosen_datasets, chosen_scores, datasets, return_ranks=True, model_indx=np.arange(len(chosen_models)))
#     chosen_ranks_all = get_ranks_s(chosen_datasets, chosen_scores, datasets, return_ranks=False, model_indx=np.arange(len(chosen_models)))
#     chosen_ranks_aggr = pd.DataFrame(columns=chosen_datasets)
#     for d in chosen_datasets:
#         chosen_ranks_aggr[d] = chosen_ranks[d].drop(columns=['model']).mean(axis=1)
#     # ranks_aggr.transpose()
#     tsf = pd.read_csv("data/datasets_features/dataset_level/tsfresh_dataset_mean.csv").sort_values("dataset").set_index("dataset").loc[chosen_datasets, :].reset_index().drop(columns=['dataset'])
#     c22 = pd.read_csv("data/datasets_features/dataset_level/catch22_dataset_mean.csv").sort_values("dataset").set_index("dataset").loc[chosen_datasets, :].reset_index().drop(columns=['dataset'])
#     mr  = pd.read_csv("data/datasets_features/dataset_level/minirocket_dataset_mean.csv").sort_values("dataset").set_index("dataset").loc[chosen_datasets, :].reset_index().drop(columns=['dataset'])
#     sm  = pd.read_csv("data/datasets_features/dataset_level/summary_dataset_mean.csv").sort_values("dataset").set_index("dataset").loc[chosen_datasets, :].reset_index().drop(columns=['dataset'])
#     lm = pd.read_csv("data/datasets_features/landmarking_raw.csv").sort_values("dataset").set_index("dataset").loc[chosen_datasets, :].reset_index().drop(columns=['dataset'])
#     tsf = tsf.dropna(axis=1)
#     tsf = tsf.drop(columns=tsf.columns[(tsf.abs() > 1e9).any()])
#     constant_features = [col for col in tsf.columns if tsf[col].nunique() <= 1]
#     tsf = tsf.drop(columns=constant_features)
#     metods_data_list = [
#         [rand_ind_method, c22.values,
#         range(2, 21),
#         False, False,
#         {},
#         False, False,
#         f'Random_30_9_F_{i}'],
        
#         # [get_more_different_datasets,
#         # c22.values,
#         # range(2, 21),
#         # False,
#         # False,
#         # {'scale_data': True},
#         # False,
#         # False,
#         # f'Cosine_Method_Catch22_30_9_F_{i}'],
        
#         [k_means_ind,
#         tsf.values,
#         range(2, 21),
#         False,
#         False,
#         {'scale_data': True},
#         False,
#         False,
#         f'K-Means_S_TSFRESH_30_9_FFFFF_{i}'],
        
#         # [get_more_different_datasets_euclid,
#         # sm.values,
#         # range(2, 21),
#         # False,
#         # False,
#         # {'scale_data': True},
#         # False,
#         # False,
#         # f'EuclidMethod_MiniRocket_30_9_F_{i}'],
#     ]
#     testing_pipeline(chosen_datasets, metods_data_list, chosen_ranks, chosen_scores, chosen_datasets, ratio=0.8, models_prefix='',
#                     load_res=True, save_results=True, save_checpoints=True, num_models=len(chosen_models))
# None

In [8]:
def auc_from_raw_csv(csv_path, metric, invert=False):

    df = pd.read_csv(csv_path)

    df_m = df[df["metric"] == metric]

    mean_by_k = (
        df_m
        .groupby("size", sort=True)["value"]
        .mean()
        .reset_index()
    )

    x = mean_by_k["size"].values.astype(float)
    y = mean_by_k["value"].values.astype(float)

    if invert:
        y = 1.0 - y

    return float(np.trapz(y, x))


def collect_aucs(root_dir, prefix):

    root_dir = Path(root_dir)

    subdirs = sorted(
        d for d in root_dir.iterdir()
        if d.is_dir() and d.name.startswith(prefix)
    )

    auc_mae = []
    auc_spearman = []
    auc_NDCG5 = []
    auc_kendall = []

    for d in subdirs:
        csv_path = d / "raw_results.csv"
        if not csv_path.exists():
            raise FileNotFoundError(csv_path)

        auc_mae.append(
            auc_from_raw_csv(csv_path, "MAE", invert=False)
        )
        auc_spearman.append(
            auc_from_raw_csv(csv_path, "Spearman", invert=True)
        )
        auc_NDCG5.append(
            auc_from_raw_csv(csv_path, "NDCG@5", invert=True)
        )
        auc_kendall.append(
            auc_from_raw_csv(csv_path, "Kendall", invert=True)
        )

    return (
        np.asarray(auc_mae, dtype=float),
        np.asarray(auc_spearman, dtype=float),
        np.asarray(auc_NDCG5, dtype=float),
        np.asarray(auc_kendall, dtype=float),
    )

In [9]:
root = Path('checkpoints\\full_testing_ch')
prefix = "EuclidMethod_MiniRocket_30_9_F_"

euc_auc_mae, euc_auc_spearman, euc_auc_ndcg5, euc_auc_kendall = collect_aucs(root, prefix)

euc_auc_mae.shape, euc_auc_spearman.shape, euc_auc_ndcg5.shape, euc_auc_ndcg5.shape

((100,), (100,), (100,), (100,))

In [10]:
root = Path('checkpoints\\full_testing_ch')
prefix = "Cosine_Method_Catch22_30_9_F_"

cos_auc_mae, cos_auc_spearman, cos_auc_ndcg5, cos_auc_kendall = collect_aucs(root, prefix)

cos_auc_mae.shape, cos_auc_spearman.shape, cos_auc_ndcg5.shape, cos_auc_ndcg5.shape

((100,), (100,), (100,), (100,))

In [11]:
root = Path('checkpoints\\full_testing_ch')
prefix = "K-Means_S_TSFRESH_30_9_FFFFF_"

kmeans_auc_mae, kmeans_auc_spearman, kmeans_auc_ndcg5, kmeans_auc_kendall = collect_aucs(root, prefix)

kmeans_auc_mae.shape, kmeans_auc_spearman.shape, kmeans_auc_ndcg5.shape, kmeans_auc_ndcg5.shape

((100,), (100,), (100,), (100,))

In [12]:
root = Path('checkpoints\\full_testing_ch')
prefix = "Random_30_9_F_"

rand_auc_mae, rand_auc_spearman, rand_auc_ndcg5, rand_auc_kendall = collect_aucs(root, prefix)

rand_auc_mae.shape, rand_auc_spearman.shape, rand_auc_ndcg5.shape, rand_auc_ndcg5.shape

((100,), (100,), (100,), (100,))

In [13]:
def win_by_auc_share(auc_method, auc_random):

    auc_method = np.asarray(auc_method, dtype=float)
    auc_random = np.asarray(auc_random, dtype=float)

    return (auc_random - auc_method) / auc_random


In [14]:
(np.mean(win_by_auc_share(cos_auc_mae, rand_auc_mae)),
np.mean(win_by_auc_share(kmeans_auc_mae, rand_auc_mae)),
np.mean(win_by_auc_share(euc_auc_mae, rand_auc_mae)),)


(0.03611949678279217, 0.020474599039852763, -0.018998782614646798)

In [15]:
(-1*np.mean(win_by_auc_share(cos_auc_spearman, rand_auc_spearman)),
-1*np.mean(win_by_auc_share(kmeans_auc_spearman, rand_auc_spearman)),
-1*np.mean(win_by_auc_share(euc_auc_spearman, rand_auc_spearman)))

(-0.09655212699220787, 0.11740395229343745, 0.09509066954808022)

In [16]:
(-1*np.mean(win_by_auc_share(cos_auc_ndcg5, rand_auc_ndcg5)),
-1*np.mean(win_by_auc_share(kmeans_auc_ndcg5, rand_auc_ndcg5)),
-1*np.mean(win_by_auc_share(euc_auc_ndcg5, rand_auc_ndcg5)))

(-0.11545217507420562, 0.15203835929696674, 0.05839127136357192)

In [17]:
(-1*np.mean(win_by_auc_share(cos_auc_kendall, rand_auc_kendall)),
-1*np.mean(win_by_auc_share(kmeans_auc_kendall, rand_auc_kendall)),
-1*np.mean(win_by_auc_share(euc_auc_kendall, rand_auc_kendall)))

(-0.0614072712109284, 0.05785489811488289, 0.06035538244226745)